### 1.
Load the movies.dat and ratings.dat files from the ml-10m dataset into two DataFrames.Merge the two DataFrames and create a column named "year"
that contains the year in which each rating was given.


In [1]:
import sys
import os

# Add the project root to the path so we can import 'src'
# Assuming our notebook is in 'recommender-ml10m/notebooks/'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_loader import DataLoader

# Initialize with a path relative to the PROJECT ROOT
# This 'data' folder will be created in our main project directory
loader = DataLoader(base_path=os.path.join(project_root, 'data'))

# Download, extract, and load the dataset
df = loader.get_processed_data()

print(f"Data shape: {df.shape}")
print(df.head())

Merging data...
Converting timestamps...
Data shape: (10000054, 6)
   userId  movieId  rating                 title  \
0       1      122     5.0      Boomerang (1992)   
1       1      185     5.0       Net, The (1995)   
2       1      231     5.0  Dumb & Dumber (1994)   
3       1      292     5.0       Outbreak (1995)   
4       1      316     5.0       Stargate (1994)   

                         genres  year  
0                Comedy|Romance  1996  
1         Action|Crime|Thriller  1996  
2                        Comedy  1996  
3  Action|Drama|Sci-Fi|Thriller  1996  
4       Action|Adventure|Sci-Fi  1996  


### 2.
Find the top 5 genres that experienced the greatest decrease in average annual rating from the first year data is available to the last.
Note that the earliest year with ratings may differ per genre.


In [2]:
from src.analyzer import GenreAnalyzer

analyzer = GenreAnalyzer(df)
summary = analyzer.analyze_trends()

# --- View 1: The Raw Exploded Data (Massive Table) ---
# Be careful with .head(), this table has ~25 million rows!
print("Exploded View Sample:")
display(analyzer.exploded_df.head(15))

# --- View 2: The Yearly Averages for EVERY genre/year combination ---
print("\nAnnual Stats Table:")
display(analyzer.annual_stats.sort_values(['genre', 'year']))

# --- View 3: The Full Results (Not just top 5) ---
print("\nTop 5 Genres with Greatest Decrease in avg annual rating:")
display(analyzer.summary_df.head(5))

# --- Insight for Top 5 Genres with Greatest Decrease  ---
# Look for genres with a small 'start_count'.
# These are the ones where the 'earliest year' likely skewed the results.

Exploding genres...
Exploded View Sample:


,year,rating,genres,genre
0,1996,5.0,Comedy|Romance,Comedy
0,1996,5.0,Comedy|Romance,Romance
1,1996,5.0,Action|Crime|Thriller,Action
1,1996,5.0,Action|Crime|Thriller,Crime
1,1996,5.0,Action|Crime|Thriller,Thriller
2,1996,5.0,Comedy,Comedy
3,1996,5.0,Action|Drama|Sci-Fi|Thriller,Action
3,1996,5.0,Action|Drama|Sci-Fi|Thriller,Drama
3,1996,5.0,Action|Drama|Sci-Fi|Thriller,Sci-Fi
3,1996,5.0,Action|Drama|Sci-Fi|Thriller,Thriller



Annual Stats Table:


,genre,year,avg_rating,rating_count
0,(no genres listed),2004,2.750000,2
1,(no genres listed),2007,4.000000,3
2,(no genres listed),2008,4.000000,2
3,Action,1995,3.000000,1
4,Action,1996,3.440509,331387
...,...,...,...,...
271,Western,2005,3.460447,23753
272,Western,2006,3.525736,14940
273,Western,2007,3.529675,11845
274,Western,2008,3.598341,12660



Top 5 Genres with Greatest Decrease in avg annual rating:


,genre,start_year,end_year,start_rating,end_rating,start_count,decrease
11,Horror,1995,2009,5.000000,3.310457,1,1.689543
17,Thriller,1995,2009,5.000000,3.472470,1,1.527530
14,Mystery,1995,2009,5.000000,3.659091,1,1.340909
6,Crime,1995,2009,4.000000,3.619338,2,0.380662
18,War,1996,2009,3.943008,3.723529,59973,0.219478


### 3.
 Is the previous analysis regarding the decline in genre popularity affected by the number of
ratings? Would you take this into account? Make an adjustment for this and comment on the results.

####
    The decline in genre popularity is absolutely affected by the number of ratings. The year a genre starts appearing in the dataset plays a massive role
    for three main reasons:
    1. Selection Bias (The "Early Adopter" Effect): In 1995–1997, users of MovieLens were primarily tech-savvy early adopters, researchers, or hardcore cinephiles.
       Their rating behavior differs significantly from the "general public" who joined in 2005.
    2. Sample Size Sensitivity: If the "Film-Noir" genre has its first rating in 1996 with only 10 total ratings, a single 1-star review will tank the average.
       By 2009, with 10,000 ratings, the average is much more stable.
    3. Regression to the Mean: Most genres start with high averages because the first people to rate a "Documentary" or "Western" in a new system are usually fans
       of that genre. As the dataset grows, more casual viewers rate those movies, typically pulling the average down toward the global mean (usually around 3.5).

To address the decline in genre popularity fairly, we need to distinguish between a genuine drop in quality/interest and statistical noise caused by low sample
sizes in the early years.

The most professional way to handle this is using a Bayesian Average (similar to how IMDb calculates Top 250 rankings). This "pulls" the average of years with
few ratings toward the global mean, preventing them from having extreme, unearned peaks or valleys.

In [9]:
adjusted_results = analyzer.get_adjusted_trends(confidence_quantile=0.1)

print("--- Comparison: Raw vs. Adjusted Decrease / Top 5 (sorted) ---")
comparison = adjusted_results[['genre', 'raw_decrease', 'adjusted_decrease', 'start_count']].sort_values(by='adjusted_decrease',ascending=False).head(5)
display(comparison)

print("--- Comparison: Raw vs. Adjusted Decrease / Top 15 (unsorted) ---")
comparison = adjusted_results[['genre', 'raw_decrease', 'adjusted_decrease', 'start_count']].head(15)
display(comparison)


--- Comparison: Raw vs. Adjusted Decrease / Top 5 (sorted) ---


,genre,raw_decrease,adjusted_decrease,start_count
18,War,0.219478,0.332954,59973
7,Documentary,0.185136,0.264646,4736
5,Comedy,-0.324105,0.159630,2
15,Romance,0.174112,0.155119,231765
3,Animation,0.116184,0.141501,56977


--- Comparison: Raw vs. Adjusted Decrease / Top 15 (unsorted) ---


,genre,raw_decrease,adjusted_decrease,start_count
18,War,0.219478,0.332954,59973
7,Documentary,0.185136,0.264646,4736
5,Comedy,-0.324105,0.159630,2
15,Romance,0.174112,0.155119,231765
3,Animation,0.116184,0.141501,56977
13,Musical,0.165284,0.135371,55378
11,Horror,1.689543,0.111569,1
4,Children,0.147742,0.095651,102193
10,Film-Noir,-0.183983,0.081216,3446
1,Action,-0.434257,0.062439,1


Commentary on Step 3:
The raw analysis in Step 2 is highly sensitive to the 'Early Year' effect. For example, if a genre had only 50 ratings in its first year (1995), a few
5-star reviews from enthusiasts would inflate the average.

By applying a Bayesian adjustment:
1. We pull 'uncertain' early averages toward the global mean (~3.5).
2. Genres that showed a massive decrease simply because they started with
   a tiny, biased sample size will see their 'adjusted_decrease' shrink.
3. The resulting 'Top 5' is now much more representative of actual
   shifts in broad audience sentiment rather than statistical outliers.

#### 4.